In [ ]:
# Requests beautiful soap pandas numpy

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

In [ ]:
import csv
import os
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup


def fetch_page_content(url):
    """Fetches the raw HTML content of a given URL with defensive error handling."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
            " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
    }
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            return response.text
        else:
            print(
                f"[Warning] Failed to connect to {url}. HTTP Status:"
                f" {response.status_code}"
            )
            return None
    except requests.exceptions.RequestException as e:
        print(f"[Error] Network exception encountered for {url}: {e}")
        return None


def parse_product_data(html_content):
    """Parses structural product attributes out of the raw HTML layout."""
    soup = BeautifulSoup(html_content, "html.parser")
    products_data = []

    # Locate individual product containers based on the page's HTML structure
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        try:
            # Extract Title from the image anchor attribute (contains the unabbreviated text)
            title = book.h3.a["title"]

            # Extract Price and clean currency text symbols for numeric readiness
            price_text = book.find("p", class_="price_color").text
            price = price_text.replace("£", "").strip()

            # Extract Availability Status
            availability_tag = book.find("p", class_="instock availability")
            availability = (
                availability_tag.text.strip() if availability_tag else "Unknown"
            )

            # Extract Star Rating (Map class text strings to numerical metrics)
            rating_classes = book.find("p", class_="star-rating")["class"]
            rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
            star_rating = rating_map.get(rating_classes[1], 0)

            products_data.append(
                {
                    "Title": title,
                    "Price_GBP": float(price),
                    "Availability": availability,
                    "Star_Rating": star_rating,
                }
            )
        except (AttributeError, TypeError, KeyError):
            # Gracefully skip single malformed records to maintain pagination continuity
            continue

    return products_data


def save_dataset(data, output_filename):
    """Appends gathered dictionary arrays into a structured, unified CSV dataset."""
    if not data:
        return

    keys = data[0].keys()
    file_exists = os.path.isfile(output_filename)

    with open(output_filename, "a", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=keys)
        if not file_exists:
            writer.writeheader()
        writer.writerows(data)


# Execution Entry Point
if __name__ == "__main__":
    # Utilizing a compliant public sandbox domain optimized for web scraping education
    base_url = "[link removed]{}.html"
    output_file = "custom_books_dataset.csv"

    # Reset tracking file if it already exists from prior workspace iterations
    if os.path.exists(output_file):
        os.remove(output_file)

    print("Status: Initializing CodeAlpha Task 1 Data Acquisition Pipeline...\n")
    total_pages_to_scrape = 5  # Navigating 5 consecutive pages to extract a tailored dataset

    for page_num in range(1, total_pages_to_scrape + 1):
        target_url = base_url.format(page_num)
        print(f"Action: Navigating page {page_num}/{total_pages_to_scrape} -> {target_url}")

        raw_html = fetch_page_content(target_url)

        if raw_html:
            page_records = parse_product_data(raw_html)
            save_dataset(page_records, output_file)
            print(f"Success: Extracted and appended {len(page_records)} data rows.")
        else:
            print(f"Failure: Skipping target page {page_num} due to network omission.")

        # Conscientious crawling compliance: throttled delay between hits
        time.sleep(1)

    print(
        f"\nPipeline Complete: Custom dataset compiled at './{output_file}'"
    )

Status: Initializing CodeAlpha Task 1 Data Acquisition Pipeline...

Action: Navigating page 1/5 -> [link removed]1.html
[Error] Network exception encountered for [link removed]1.html: Failed to parse: [link removed]1.html
Failure: Skipping target page 1 due to network omission.
Action: Navigating page 2/5 -> [link removed]2.html
[Error] Network exception encountered for [link removed]2.html: Failed to parse: [link removed]2.html
Failure: Skipping target page 2 due to network omission.
Action: Navigating page 3/5 -> [link removed]3.html
[Error] Network exception encountered for [link removed]3.html: Failed to parse: [link removed]3.html
Failure: Skipping target page 3 due to network omission.
Action: Navigating page 4/5 -> [link removed]4.html
[Error] Network exception encountered for [link removed]4.html: Failed to parse: [link removed]4.html
Failure: Skipping target page 4 due to network omission.
Action: Navigating page 5/5 -> [link removed]5.html
[Error] Network exception encountere

In [ ]:
# Verify dataset compilation parameters using Pandas dataframes
if os.path.exists(output_file):
    df = pd.read_csv(output_file)

    print("=== DATASET STRUCTURAL VERIFICATION ===")
    print(f"Total Collected Rows : {df.shape[0]}")
    print(f"Total Feature Columns: {df.shape[1]}\n")

    print("=== HEAD DATAFRAME PREVIEW ===")
    display(df.head(10))
else:
    print("[Error] Compiled dataset could not be located in the local directory workspace.")

[Error] Compiled dataset could not be located in the local directory workspace.
